In [ ]:
!pip install "numpy<2.0" --force-reinstall
!pip install "pillow<10.0" --force-reinstall
!pip install "matplotlib<3.8" --force-reinstall
!pip install --no-cache-dir ultralytics


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi

In [7]:
# Code to delete files from output folder
import shutil
import os

working_dir = "/kaggle/working"

for item in os.listdir(working_dir):
    item_path = os.path.join(working_dir, item)
    if os.path.isfile(item_path):
        os.remove(item_path)
    else:
        shutil.rmtree(item_path)

print("All output files deleted.")


All output files deleted.


In [8]:
import os
from ultralytics import YOLO

ROOT_DIR = "/kaggle/input/license-plate-recognition/License Plate Recognition"

# Load pretrained YOLOv8s
model = YOLO("yolov8s.pt")

results = model.train(
    data=os.path.join(ROOT_DIR, "data.yaml"),
    
    # Training schedule
    epochs=70,
    patience=15,              # improved early stopping
    # Hardware settings
    device="0,1",
    imgsz=640,
    batch=32,                 # optimal for T4/P100 (falls back to 16 automatically if RAM is low)
    # Speed / stability improvements
    cache=True,               # MUCH faster dataloading
    amp=True,                 # automatic mixed precision
    optimizer="AdamW",        # faster convergence
    lr0=0.001,                # stable LR    
    # Project settings
    name="yolov8_trained_model",
    project=".",
)


Ultralytics 8.3.228 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
                                                       CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/license-plate-recognition/License Plate Recognition/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_tr

In [11]:
import cv2
from pathlib import Path

# Paths
MODEL_PATH = "/kaggle/working/yolov8_trained_model/weights/best.pt"
TEST_DIR = "/kaggle/input/license-plate-recognition/License Plate Recognition/test/images"  # put your test images here
OUTPUT_DIR = "/kaggle/working/test_results"

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load trained YOLOv8 model
model = YOLO(MODEL_PATH)

# Predict on all images in TEST_DIR
test_images = list(Path(TEST_DIR).glob("*.*"))  # all image files

print(f"Found {len(test_images)} images for inference.")

for img_path in test_images:
    # Run prediction
    results = model.predict(
        source=str(img_path),
        imgsz=640,         # same size as training
        conf=0.25,         # confidence threshold
        iou=0.45,          # NMS IoU threshold
        device="0,1",          # GPU 0 on Kaggle
        save=False         # we will save annotated images manually
    )
    
    # Annotate
    annotated_img = results[0].plot()  # returns numpy array

    # Convert RGB to BGR for OpenCV
    annotated_img_bgr = cv2.cvtColor(annotated_img, cv2.COLOR_RGB2BGR)

    # Save
    output_path = os.path.join(OUTPUT_DIR, img_path.name)
    cv2.imwrite(output_path, annotated_img_bgr)

print(f"Inference complete! Annotated images saved in: {OUTPUT_DIR}")

Found 1019 images for inference.

image 1/1 /kaggle/input/license-plate-recognition/License Plate Recognition/test/images/xemay1040_jpg.rf.15d5cf1aac8fd10e05c615ba052bb311.jpg: 640x640 1 License_Plate, 16.2ms
Speed: 2.1ms preprocess, 16.2ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /kaggle/input/license-plate-recognition/License Plate Recognition/test/images/xemay281_jpg.rf.65c06b51fd9e42a82cdcc80b7f6f7d7b.jpg: 640x640 1 License_Plate, 16.2ms
Speed: 1.8ms preprocess, 16.2ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /kaggle/input/license-plate-recognition/License Plate Recognition/test/images/0269b68e1093b0be_jpg.rf.29fd59c72a78dd9e2406c280d07503d3.jpg: 640x640 1 License_Plate, 16.2ms
Speed: 1.7ms preprocess, 16.2ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /kaggle/input/license-plate-recognition/License Plate Recognition/test/images/CarLongPlateGen3902_jpg.rf.1c4426118cfa2a442080704962

In [17]:
from zipfile import ZipFile, ZIP_DEFLATED

WORKING_DIR = "/kaggle/working/yolov8_trained_model"
ZIP_PATH = "/kaggle/working/working_folder.zip"

with ZipFile(ZIP_PATH, 'w', compression=ZIP_DEFLATED, allowZip64=True) as zipf:
    for root, dirs, files in os.walk(WORKING_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            # Keep folder structure relative to WORKING_DIR
            arcname = os.path.relpath(file_path, start=WORKING_DIR)
            zipf.write(file_path, arcname)

print(f"All files in {WORKING_DIR} zipped at: {ZIP_PATH}")


All files in /kaggle/working/yolov8_trained_model zipped at: /kaggle/working/working_folder.zip
